# NB01 (self-contained) — data preparation + training in ONE notebook
**Settings:** GPU T4 x2 · Internet On · **Inputs: the same 5 datasets as NB00** (isic-2019, ham1000-segmentation-and-classification,
skin-cancer (PAD-UFES-20), ph2dataset, derm7pt). No notebook outputs needed.

Step 1 prepares `/kaggle/working/prep` (identical code to NB00 v2, ~50 min; skipped if it already exists).
Step 2 trains. First run interactively with `SMOKE = 1`, then set `SMOKE = 0` and *Save & Run All*.

In [ ]:
%%writefile artproto.py
"""
artproto.py -- shared library for the paper
"Artifact-aware prototype learning for skin-lesion classification".

Everything (data caching, lesion/artifact masks, models, losses, training,
push, evaluation, counterfactuals) lives here so that every notebook uses
exactly the same code.
"""
import os, re, json, glob, math, time, random, copy, traceback
import numpy as np
import pandas as pd
import cv2
import torch
import torch.nn as nn
import torch.nn.functional as F

IMG = 256          # cached resolution (square, as in the original project)
OUT = 224          # network input resolution
MEAN = (0.485, 0.456, 0.406)
STD = (0.229, 0.224, 0.225)

# bit layout of the uint8 mask array
BIT_L, BIT_DC, BIT_TS, BIT_SP, BIT_INK = 1, 2, 4, 8, 16
BIT_ART = BIT_DC | BIT_TS | BIT_SP | BIT_INK
ART_BITS = {"DC": BIT_DC, "TS": BIT_TS, "SP": BIT_SP, "INK": BIT_INK}
ART_LONG = {"DC": "dark corner / vignette", "TS": "thin dark structures (hair, ruler ticks)",
            "SP": "specular highlights / gel bubbles", "INK": "ink markings / coloured patches"}
DETECTOR_VERSION = 2
# image-level presence thresholds (fraction of image area)
ART_MIN_FRAC = {"DC": 0.01, "TS": 0.003, "SP": 0.0005, "INK": 0.001}


# ----------------------------------------------------------------------------- utils
def set_seed(s):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)


def walk_files(root="/kaggle/input"):
    """Recursive file list. followlinks=True is REQUIRED: recent Kaggle images mount
    datasets as symlinks under /kaggle/input, which os.walk skips by default."""
    out, seen = [], set()
    for r, ds, fs in os.walk(root, followlinks=True):
        real = os.path.realpath(r)
        if real in seen:                      # guard against symlink cycles
            ds[:] = []; continue
        seen.add(real)
        for f in fs:
            out.append(os.path.join(r, f).replace(os.sep, '/'))
    return out


def load_rgb_resized(path, size=IMG):
    from PIL import Image
    im = Image.open(path)
    try:
        im.draft("RGB", (size * 2, size * 2))      # fast JPEG decoding at reduced scale
    except Exception:
        pass
    a = np.asarray(im.convert("RGB"))
    return cv2.resize(a, (size, size), interpolation=cv2.INTER_AREA)


def load_mask_resized(path, size=IMG):
    from PIL import Image
    m = np.asarray(Image.open(path).convert("L"))
    m = cv2.resize(m, (size, size), interpolation=cv2.INTER_AREA)
    return (m >= 128).astype(np.uint8)


def phash64(img):
    g = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY).astype(np.float32)
    g = cv2.resize(g, (32, 32), interpolation=cv2.INTER_AREA)
    d = cv2.dct(g)[:8, :8].flatten()
    return d > np.median(d[1:])


def thumbs(img):
    """z-normalised 32x32 grey thumbnails of the full image and of the central 50% crop
    (used to VERIFY pHash duplicate candidates; the centre crop ignores vignettes)."""
    g = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY).astype(np.float32)
    out = []
    for a in (g, g[64:192, 64:192]):
        t = cv2.resize(a, (32, 32), interpolation=cv2.INTER_AREA).flatten()
        t = (t - t.mean()) / (t.std() + 1e-6) / np.sqrt(t.size)
        out.append(t.astype(np.float16))
    return out[0], out[1]


def hamming_pairs(bitsA, bitsB=None, thr=6, device="cuda", chunk=2048):
    """Return (i, j, ham) with ham <= thr. If bitsB is None: pairs inside A with i<j."""
    dev = device if torch.cuda.is_available() else "cpu"
    dt = torch.float16 if dev != "cpu" else torch.float32
    A = torch.from_numpy(bitsA.astype(np.float32) * 2 - 1).to(dev, dt)
    B = A if bitsB is None else torch.from_numpy(bitsB.astype(np.float32) * 2 - 1).to(dev, dt)
    out = []
    for s in range(0, len(A), chunk):
        ham = (64 - (A[s:s + chunk] @ B.T).float()) / 2
        ii, jj = torch.nonzero(ham <= thr + 1e-3, as_tuple=True)
        h = ham[ii, jj]
        ii = ii + s
        if bitsB is None:
            keep = ii < jj
            ii, jj, h = ii[keep], jj[keep], h[keep]
        out.append(torch.stack([ii.float(), jj.float(), h], 1).cpu())
    o = torch.cat(out).numpy() if out else np.zeros((0, 3))
    return o[:, 0].astype(int), o[:, 1].astype(int), o[:, 2].astype(int)


class UnionFind:
    def __init__(self, n): self.p = np.arange(n)
    def find(self, a):
        while self.p[a] != a:
            self.p[a] = self.p[self.p[a]]; a = self.p[a]
        return a
    def union(self, a, b):
        ra, rb = self.find(a), self.find(b)
        if ra != rb: self.p[max(ra, rb)] = min(ra, rb)


# ----------------------------------------------------------------------------- masks
def postprocess_lesion(prob, thr=0.5, min_frac=0.005):
    m = (prob >= thr).astype(np.uint8)
    n, lab, st, _ = cv2.connectedComponentsWithStats(m, 8)
    if n <= 1:
        return np.zeros_like(m), False
    j = 1 + int(np.argmax(st[1:, cv2.CC_STAT_AREA]))
    m = (lab == j).astype(np.uint8)
    inv = 1 - m                                         # fill holes
    n2, lab2, _, _ = cv2.connectedComponentsWithStats(inv, 4)
    border = np.unique(np.concatenate([lab2[0], lab2[-1], lab2[:, 0], lab2[:, -1]]))
    holes = ~np.isin(lab2, border) & (inv > 0)
    m[holes] = 1
    return m, bool(m.mean() >= min_frac)


def otsu_lesion(img):
    """Classical fallback lesion segmentation (only used if no GT masks are available)."""
    b = cv2.GaussianBlur(img[..., 2], (11, 11), 0)
    _, m = cv2.threshold(b, 0, 1, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    return postprocess_lesion(m.astype(np.float32))


def detect_artifacts(img, L):
    """Rule-based artifact masks on a 256x256 RGB uint8 image.
    L: uint8 lesion mask. Artifact pixels are never inside the (dilated) lesion.
    Returns uint8 bitfield (DC|TS|SP|INK)."""
    H, W = L.shape
    area = H * W
    hsv = cv2.cvtColor(img, cv2.COLOR_RGB2HSV)
    Hh, S, V = hsv[..., 0], hsv[..., 1], hsv[..., 2]
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    Ld = cv2.dilate(L.astype(np.uint8), cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7))) > 0

    # (1) dark corners / vignette: dark regions connected to the image border
    dark = cv2.morphologyEx((V < 50).astype(np.uint8), cv2.MORPH_OPEN, np.ones((3, 3), np.uint8))
    n, lab, st, _ = cv2.connectedComponentsWithStats(dark, 8)
    border = np.unique(np.concatenate([lab[0], lab[-1], lab[:, 0], lab[:, -1]]))
    # must be a large blob (area) and not a thin line touching the border (hair): mean thickness >= 8 px
    keep = [b for b in border if b != 0 and st[b, cv2.CC_STAT_AREA] >= 0.002 * area and
            st[b, cv2.CC_STAT_AREA] / max(np.hypot(st[b, cv2.CC_STAT_WIDTH], st[b, cv2.CC_STAT_HEIGHT]), 1) >= 8]
    dc = np.isin(lab, keep) & ~Ld

    # (2) thin dark structures (hair, ruler ticks/lines): black-hat + elongation filter
    k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (13, 13))
    bh = cv2.morphologyEx(cv2.GaussianBlur(gray, (3, 3), 0), cv2.MORPH_BLACKHAT, k)
    ts = (bh >= 18).astype(np.uint8)
    ts[cv2.dilate(dc.astype(np.uint8), np.ones((7, 7), np.uint8)) > 0] = 0
    ts[:4, :] = 0; ts[-4:, :] = 0; ts[:, :4] = 0; ts[:, -4:] = 0     # v2: no frame-edge strips
    n, lab, st, _ = cv2.connectedComponentsWithStats(ts, 8)
    a = st[:, cv2.CC_STAT_AREA].astype(np.float64)
    diag = np.hypot(st[:, cv2.CC_STAT_WIDTH], st[:, cv2.CC_STAT_HEIGHT])
    good = (a >= 15) & (diag >= 15) & (a / np.maximum(diag, 1) <= 6.0)
    good[0] = False
    ts = good[lab] & ~Ld

    # (3) specular highlights / gel bubbles: small bright unsaturated blobs that are much brighter
    #     than their surround (v2: white top-hat keeps only bright peaks smaller than 11 px -> large
    #     whitish skin areas are rejected; bubble rims and glints are kept)
    tophat = cv2.morphologyEx(V, cv2.MORPH_TOPHAT, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (11, 11)))
    sp = ((V >= 210) & (S <= 50) & (tophat >= 25)).astype(np.uint8)
    n, lab, st, _ = cv2.connectedComponentsWithStats(sp, 8)
    a = st[:, cv2.CC_STAT_AREA]
    good = (a >= 3) & (a <= 0.01 * area)
    good[0] = False
    sp = (cv2.dilate(good[lab].astype(np.uint8), np.ones((3, 3), np.uint8)) > 0) & ~Ld

    # (4) ink markings / coloured patches: strongly saturated blue-cyan-violet regions outside the lesion
    #     (v2: hue 90-150 also covers blue/cyan stickers; S>=90 rejects bluish-tinted skin; pink skin
    #      (H>150) excluded; a 12-px band next to the vignette is excluded (chromatic fringe))
    ink = cv2.morphologyEx(((Hh >= 90) & (Hh <= 150) & (S >= 90) & (V >= 40) & (V <= 235)).astype(np.uint8),
                           cv2.MORPH_OPEN, np.ones((3, 3), np.uint8))
    n, lab, st, _ = cv2.connectedComponentsWithStats(ink, 8)
    good = st[:, cv2.CC_STAT_AREA] >= 40
    good[0] = False
    ink = good[lab] & ~Ld & ~(cv2.dilate(dc.astype(np.uint8), np.ones((25, 25), np.uint8)) > 0)

    return (dc * BIT_DC | ts * BIT_TS | sp * BIT_SP | ink * BIT_INK).astype(np.uint8)


def artifact_flags(mask_bits):
    """image-level presence flags from a (256,256) bitfield."""
    return {k: float(((mask_bits & b) > 0).mean()) >= ART_MIN_FRAC[k] for k, b in ART_BITS.items()}


# ----------------------------------------------------------------------------- GPU augmentation
def gpu_augment(img_u8, msk_u8, mode, out=OUT, gen=None):
    """img_u8 (B,256,256,3) uint8, msk_u8 (B,256,256) uint8 (on device).
    mode: 'eval' | 'std' | 'robust'. Image and masks receive the SAME geometric transform.
    Returns normalized image (B,3,out,out), lesion mask, artifact mask (B,1,out,out) in [0,1]."""
    dev = img_u8.device
    B = img_u8.shape[0]
    x = img_u8.permute(0, 3, 1, 2).float() / 255.0
    m = msk_u8.long()
    mL = ((m & BIT_L) > 0).float()[:, None]
    mA = ((m & BIT_ART) > 0).float()[:, None]
    R = lambda *s: torch.rand(*s, device=dev, generator=gen)
    theta = torch.zeros(B, 2, 3, device=dev)
    if mode == "eval":
        theta[:, 0, 0] = 1; theta[:, 1, 1] = 1
    elif mode == "std":       # Resize(256) + RandomCrop(224) + HFlip
        s = out / IMG
        fx = torch.where(R(B) < 0.5, -1.0, 1.0)
        theta[:, 0, 0] = s * fx; theta[:, 1, 1] = s
        theta[:, 0, 2] = (R(B) * 2 - 1) * (1 - s); theta[:, 1, 2] = (R(B) * 2 - 1) * (1 - s)
    elif mode == "robust":    # RandomResizedCrop(scale .4-1, ratio 3/4-4/3) + H/V flips (+ colour jitter)
        area = 0.4 + 0.6 * R(B)
        logr = math.log(3 / 4) + (math.log(4 / 3) - math.log(3 / 4)) * R(B)
        r = torch.exp(logr)
        sx = torch.sqrt(area * r).clamp(max=1.0); sy = torch.sqrt(area / r).clamp(max=1.0)
        fx = torch.where(R(B) < 0.5, -1.0, 1.0); fy = torch.where(R(B) < 0.5, -1.0, 1.0)
        theta[:, 0, 0] = sx * fx; theta[:, 1, 1] = sy * fy
        theta[:, 0, 2] = (R(B) * 2 - 1) * (1 - sx); theta[:, 1, 2] = (R(B) * 2 - 1) * (1 - sy)
    else:
        raise ValueError(mode)
    grid = F.affine_grid(theta, (B, 3, out, out), align_corners=False)
    x = F.grid_sample(x, grid, mode="bilinear", padding_mode="border", align_corners=False)
    mm = F.grid_sample(torch.cat([mL, mA], 1), grid, mode="bilinear", padding_mode="border", align_corners=False)
    mL, mA = mm[:, :1], mm[:, 1:]
    if mode == "robust":      # ColorJitter(brightness=.2, contrast=.2, saturation=.2)
        b = (0.8 + 0.4 * R(B)).view(B, 1, 1, 1)
        x = (x * b).clamp(0, 1)
        c = (0.8 + 0.4 * R(B)).view(B, 1, 1, 1)
        g = (0.299 * x[:, 0] + 0.587 * x[:, 1] + 0.114 * x[:, 2]).mean((1, 2)).view(B, 1, 1, 1)
        x = ((x - g) * c + g).clamp(0, 1)
        s = (0.8 + 0.4 * R(B)).view(B, 1, 1, 1)
        gy = (0.299 * x[:, 0] + 0.587 * x[:, 1] + 0.114 * x[:, 2])[:, None]
        x = (gy + (x - gy) * s).clamp(0, 1)
    mean = torch.tensor(MEAN, device=dev).view(1, 3, 1, 1)
    std = torch.tensor(STD, device=dev).view(1, 3, 1, 1)
    return (x - mean) / std, mL, mA


# ----------------------------------------------------------------------------- models
def densenet_features():
    from torchvision.models import densenet121, DenseNet121_Weights
    return densenet121(weights=DenseNet121_Weights.IMAGENET1K_V1).features


class DenseCNN(nn.Module):
    """Black-box reference: DenseNet-121 + GAP + linear."""
    kind = "cnn"
    def __init__(self, n_classes=2, pretrained=True):
        super().__init__()
        self.features = densenet_features() if pretrained else _tiny_features()
        self.head = nn.Linear(1024, n_classes)
    def forward(self, x):
        f = F.relu(self.features(x))
        return {"logits": self.head(f.mean((2, 3))), "fmap": f}


class ProtoPNet(nn.Module):
    """ProtoPNet (Chen et al., NeurIPS 2019) with a DenseNet-121 backbone."""
    kind = "proto"
    def __init__(self, n_classes=2, ppc=10, D=128, pretrained=True):
        super().__init__()
        self.features = densenet_features() if pretrained else _tiny_features()
        self.add_on = nn.Sequential(nn.Conv2d(1024, D, 1), nn.ReLU(), nn.Conv2d(D, D, 1), nn.Sigmoid())
        P = n_classes * ppc
        self.prototypes = nn.Parameter(torch.rand(P, D, 1, 1))
        self.register_buffer("proto_class", torch.arange(P) // ppc)
        ident = F.one_hot(torch.arange(P) // ppc, n_classes).float().T          # (C,P)
        self.register_buffer("class_identity", ident)
        self.last = nn.Linear(P, n_classes, bias=False)
        with torch.no_grad():
            self.last.weight.copy_(1.0 * ident - 0.5 * (1 - ident))
        self.n_classes, self.P, self.D = n_classes, P, D

    def latent(self, x):
        f = F.relu(self.features(x))
        with torch.autocast("cuda", enabled=False):
            return self.add_on(f.float())

    def distances(self, z):
        x2 = (z ** 2).sum(1, keepdim=True)
        p2 = (self.prototypes ** 2).sum((1, 2, 3)).view(1, -1, 1, 1)
        xp = F.conv2d(z, self.prototypes)
        return F.relu(x2 + p2 - 2 * xp)

    def forward(self, x):
        z = self.latent(x)
        with torch.autocast("cuda", enabled=False):
            d = self.distances(z)
            sim = torch.log((d + 1) / (d + 1e-4))
            act = sim.amax((2, 3))
            logits = self.last(act)
        return {"logits": logits, "sim": sim, "dist": d, "min_d": d.amin((2, 3)), "act": act, "z": z}


def _tiny_features():  # CPU unit tests only
    return nn.Sequential(nn.Conv2d(3, 1024, 32, stride=32))


def build_model(cfg, pretrained=True):
    if cfg["model"] == "cnn":
        return DenseCNN(pretrained=pretrained)
    return ProtoPNet(ppc=cfg.get("ppc", 10), D=cfg.get("D", 128), pretrained=pretrained)


# ----------------------------------------------------------------------------- losses
def proto_base_losses(out, y, model, class_w):
    ce = F.cross_entropy(out["logits"], y, weight=class_w)
    md = out["min_d"]
    own = model.proto_class[None, :] == y[:, None]
    big = torch.finfo(md.dtype).max / 4
    clst = md.masked_fill(~own, big).amin(1).mean()
    sep = md.masked_fill(own, big).amin(1).mean()
    return ce, clst, sep


def artifact_lesion_losses(out, y, model, cL, cA, valid, tau=0.5):
    """Proposed losses.  sim: (B,P,h,w); cL/cA: lesion/artifact fraction per latent cell (B,1,h,w).
    L_art : activation-weighted spatial-attention mass that every prototype puts on artifact cells.
    L_les : -log attention mass that own-class prototypes put on lesion cells."""
    sim = out["sim"]
    B, P, h, w = sim.shape
    att = F.softmax(sim.flatten(2) / tau, dim=2)                         # (B,P,hw)
    wt = sim.flatten(2).amax(2).detach()
    wt = wt / wt.sum(1, keepdim=True).clamp_min(1e-8)
    a_art = (att * cA.flatten(2)).sum(2)                                  # (B,P)
    l_art_i = (wt * a_art).sum(1)
    own = (model.proto_class[None, :] == y[:, None]).float()
    w_own = wt * own
    w_own = w_own / w_own.sum(1, keepdim=True).clamp_min(1e-8)
    a_les = (att * cL.flatten(2)).sum(2)
    l_les_i = (w_own * -torch.log(a_les + 1e-4)).sum(1)
    has_les = valid & (cL.flatten(1).sum(1) >= 0.5)
    zero = sim.sum() * 0
    l_art = l_art_i[valid].mean() if valid.any() else zero
    l_les = l_les_i[has_les].mean() if has_les.any() else zero
    return l_art, l_les, a_art.detach(), a_les.detach()


def fine_annotation_loss(out, y, model, cL, valid):
    """IAIA-BL style fine-annotation loss (Barnett et al., 2021) adapted to lesion masks:
    own-class prototypes are penalised for activating outside the lesion, other-class
    prototypes for activating anywhere."""
    sim = out["sim"]
    own = model.proto_class[None, :] == y[:, None]
    t_own = (sim * (1 - cL)).flatten(2).norm(dim=2)
    t_oth = sim.flatten(2).norm(dim=2)
    per = (t_own * own).sum(1) + (t_oth * ~own).sum(1)
    return per[valid].mean() if valid.any() else sim.sum() * 0


def cell_fractions(mL, mA, hw):
    return F.adaptive_avg_pool2d(mL, hw), F.adaptive_avg_pool2d(mA, hw)


# ----------------------------------------------------------------------------- data
class ArrDS(torch.utils.data.Dataset):
    def __init__(self, img_path, msk_path, labels, valid, idx):
        self.img_path, self.msk_path = img_path, msk_path
        self.labels = np.asarray(labels).astype(np.int64)
        self.valid = np.asarray(valid).astype(bool)
        self.idx = np.asarray(idx)
        self.imgs = self.msks = None
    def __len__(self): return len(self.idx)
    def __getitem__(self, k):
        if self.imgs is None:
            self.imgs = np.load(self.img_path, mmap_mode="r")
            self.msks = np.load(self.msk_path, mmap_mode="r")
        i = int(self.idx[k])
        return (torch.from_numpy(np.ascontiguousarray(self.imgs[i])),
                torch.from_numpy(np.ascontiguousarray(self.msks[i])),
                int(self.labels[i]), bool(self.valid[i]), i)


def make_loader(ds, bs, shuffle, seed=0, workers=4, drop_last=False):
    g = torch.Generator(); g.manual_seed(seed)
    return torch.utils.data.DataLoader(ds, batch_size=bs, shuffle=shuffle, generator=g, num_workers=workers,
                                       pin_memory=torch.cuda.is_available(), drop_last=drop_last,
                                       persistent_workers=workers > 0)


# ----------------------------------------------------------------------------- inference
@torch.no_grad()
def predict(model, loader, device, keep_maps=False):
    model.eval()
    probs, ys, ids, maps, acts = [], [], [], [], []
    for img, msk, y, v, i in loader:
        x, _, _ = gpu_augment(img.to(device, non_blocking=True), msk.to(device, non_blocking=True), "eval")
        with torch.autocast("cuda", dtype=torch.float16, enabled=device.type == "cuda"):
            out = model(x.contiguous(memory_format=torch.channels_last))
        probs.append(F.softmax(out["logits"].float(), 1)[:, 1].cpu()); ys.append(y); ids.append(i)
        if keep_maps and "sim" in out:
            maps.append(out["sim"].half().cpu()); acts.append(out["act"].cpu())
    r = {"prob": torch.cat(probs).numpy(), "y": torch.cat(ys).numpy(), "idx": torch.cat(ids).numpy()}
    if keep_maps and maps:
        r["sim"] = torch.cat(maps).numpy(); r["act"] = torch.cat(acts).numpy()
    return r


def auc_safe(y, p):
    from sklearn.metrics import roc_auc_score
    try:
        return float(roc_auc_score(y, p))
    except Exception:
        return float("nan")


# ----------------------------------------------------------------------------- push
@torch.no_grad()
def push_prototypes(model, loader, device, constrained, les_min=0.5, art_max=0.05):
    """Project every prototype onto its nearest latent patch of a training image of its class.
    constrained=True: candidate patches must lie on the lesion (lesion fraction >= les_min)
    and be artifact-free (artifact fraction <= art_max)."""
    model.eval()
    P, D = model.P, model.D
    best_d = torch.full((P,), float("inf"), device=device)
    best_z = model.prototypes.data.view(P, D).clone()
    info = [dict(proto=j, cls=int(model.proto_class[j]), img=-1, h=-1, w=-1, dist=float("inf"),
                 les_frac=float("nan"), art_frac=float("nan")) for j in range(P)]
    for img, msk, y, v, ii in loader:
        img, msk, y, v = img.to(device), msk.to(device), y.to(device), v.to(device)
        x, mL, mA = gpu_augment(img, msk, "eval")
        with torch.autocast("cuda", dtype=torch.float16, enabled=device.type == "cuda"):
            z = model.latent(x.contiguous(memory_format=torch.channels_last))
        d = model.distances(z.float())                                    # (B,P,h,w)
        B, _, h, w = d.shape
        cL, cA = cell_fractions(mL, mA, (h, w))
        ok = torch.ones(B, 1, h, w, dtype=torch.bool, device=device)
        if constrained:
            ok = (cL >= les_min) & (cA <= art_max) & v.view(B, 1, 1, 1)
        own = model.proto_class.view(1, P, 1, 1) == y.view(B, 1, 1, 1)
        dd = d.masked_fill(~(own & ok), float("inf")).permute(1, 0, 2, 3).reshape(P, -1)
        val, arg = dd.min(1)
        upd = val < best_d
        for j in torch.nonzero(upd).flatten().tolist():
            a = int(arg[j]); b, rem = divmod(a, h * w); hh, ww = divmod(rem, w)
            best_d[j] = val[j]; best_z[j] = z[b, :, hh, ww].float()
            info[j].update(img=int(ii[b]), h=hh, w=ww, dist=float(val[j]),
                           les_frac=float(cL[b, 0, hh, ww]), art_frac=float(cA[b, 0, hh, ww]))
    model.prototypes.data.copy_(best_z.view(P, D, 1, 1))
    return pd.DataFrame(info)


# ----------------------------------------------------------------------------- training
DEFAULT = dict(model="proto", aug="robust", lam_clst=0.8, lam_sep=0.08, lam_l1=1e-4,
               lam_art=0.0, lam_les=0.0, lam_fa=0.0, tau=0.5, push="standard",
               ep_warm=5, ep_joint=15, ep_ll=5, bs=64, lr_feat=1e-4, lr_addon=3e-3, lr_proto=3e-3,
               lr_head=1e-3, lr_ll=1e-3, wd=1e-3, seed=0)

CONFIGS = {
    "CNN":         dict(model="cnn"),
    "PPN-std":     dict(aug="std"),
    "PPN":         dict(),
    "PPN-FA":      dict(lam_fa=1e-3),
    "Ours":        dict(lam_art=1.0, lam_les=1.0, push="constrained"),
    "Ours-noLes":  dict(lam_art=1.0, lam_les=0.0, push="constrained"),
    "Ours-noArt":  dict(lam_art=0.0, lam_les=1.0, push="constrained"),
    "Ours-stdPush": dict(lam_art=1.0, lam_les=1.0, push="standard"),
}


def full_cfg(name, seed):
    c = dict(DEFAULT); c.update(CONFIGS[name]); c["name"] = name; c["seed"] = seed
    return c


def _set_trainable(model, stage):
    for p in model.parameters(): p.requires_grad = False
    if model.kind == "cnn":
        for p in model.head.parameters(): p.requires_grad = True
        if stage == "joint":
            for p in model.features.parameters(): p.requires_grad = True
        return
    if stage in ("warm", "joint"):
        for p in model.add_on.parameters(): p.requires_grad = True
        model.prototypes.requires_grad = True
        if stage == "joint":
            for p in model.features.parameters(): p.requires_grad = True
    elif stage == "last":
        for p in model.last.parameters(): p.requires_grad = True


def train_run(cfg, prep, outdir, device, smoke=False, workers=4, log=print):
    """prep: dict(img_path, msk_path, meta DataFrame with split,label,lesion_valid)."""
    os.makedirs(outdir, exist_ok=True)
    set_seed(cfg["seed"])
    torch.backends.cudnn.benchmark = True
    meta = prep["meta"]
    tr = np.where(meta.split.values == "train")[0]
    va = np.where(meta.split.values == "val")[0]
    if smoke:
        rs = np.random.RandomState(0)
        tr, va = rs.choice(tr, min(512, len(tr)), replace=False), rs.choice(va, min(256, len(va)), replace=False)
        cfg = dict(cfg, ep_warm=1, ep_joint=1, ep_ll=1)
    labels, valid = meta.label.values, meta.lesion_valid.values
    mk = lambda idx: ArrDS(prep["img_path"], prep["msk_path"], labels, valid, idx)
    tr_loader = make_loader(mk(tr), cfg["bs"], True, cfg["seed"], workers, drop_last=True)
    tr_eval_loader = make_loader(mk(tr), 128, False, 0, workers)
    va_loader = make_loader(mk(va), 128, False, 0, workers)
    n1 = labels[tr].sum(); n0 = len(tr) - n1
    class_w = torch.tensor([len(tr) / (2 * n0), len(tr) / (2 * n1)], dtype=torch.float32, device=device)

    model = build_model(cfg, pretrained=prep.get("pretrained", True)).to(device).to(memory_format=torch.channels_last)
    is_proto = model.kind == "proto"
    if is_proto:
        opt_warm = torch.optim.Adam([{"params": model.add_on.parameters(), "lr": cfg["lr_addon"], "weight_decay": cfg["wd"]},
                                     {"params": [model.prototypes], "lr": cfg["lr_proto"]}])
        opt_joint = torch.optim.Adam([{"params": model.features.parameters(), "lr": cfg["lr_feat"], "weight_decay": cfg["wd"]},
                                      {"params": model.add_on.parameters(), "lr": cfg["lr_addon"], "weight_decay": cfg["wd"]},
                                      {"params": [model.prototypes], "lr": cfg["lr_proto"]}])
    else:
        opt_warm = torch.optim.Adam(model.head.parameters(), lr=cfg["lr_head"])
        opt_joint = torch.optim.Adam([{"params": model.features.parameters(), "lr": cfg["lr_feat"], "weight_decay": cfg["wd"]},
                                      {"params": model.head.parameters(), "lr": cfg["lr_head"]}])
    sched = torch.optim.lr_scheduler.StepLR(opt_joint, step_size=5, gamma=0.1)
    scaler = torch.amp.GradScaler("cuda", enabled=device.type == "cuda")
    hist, best = [], dict(auc=-1, state=None, epoch=-1)
    E = cfg["ep_warm"] + cfg["ep_joint"]
    t_start = time.time()

    def one_epoch(opt, stage):
        # last-layer stage: everything below the last layer is frozen INCLUDING BatchNorm statistics,
        # so the latent space (and hence the pushed prototypes = real training patches) stays exact.
        model.train(stage != "last")
        agg = {}
        for img, msk, y, v, ii in tr_loader:
            img, msk = img.to(device, non_blocking=True), msk.to(device, non_blocking=True)
            y, v = y.to(device), v.to(device)
            x, mL, mA = gpu_augment(img, msk, cfg["aug"])
            with torch.autocast("cuda", dtype=torch.float16, enabled=device.type == "cuda"):
                out = model(x.contiguous(memory_format=torch.channels_last))
            parts = {}
            if is_proto:
                ce, clst, sep = proto_base_losses(out, y, model, class_w)
                parts.update(ce=ce, clst=clst, sep=sep)
                if stage == "last":
                    l1 = (model.last.weight * (1 - model.class_identity)).norm(p=1)
                    loss = ce + cfg["lam_l1"] * l1
                else:
                    loss = ce + cfg["lam_clst"] * clst - cfg["lam_sep"] * sep
                    if cfg["lam_art"] > 0 or cfg["lam_les"] > 0:
                        cL, cA = cell_fractions(mL, mA, out["sim"].shape[2:])
                        la, ll, _, _ = artifact_lesion_losses(out, y, model, cL, cA, v, cfg["tau"])
                        loss = loss + cfg["lam_art"] * la + cfg["lam_les"] * ll
                        parts.update(art=la, les=ll)
                    if cfg["lam_fa"] > 0:
                        cL, _ = cell_fractions(mL, mA, out["sim"].shape[2:])
                        fa = fine_annotation_loss(out, y, model, cL, v)
                        loss = loss + cfg["lam_fa"] * fa
                        parts.update(fa=fa)
            else:
                loss = F.cross_entropy(out["logits"].float(), y, weight=class_w)
                parts.update(ce=loss)
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.step(opt); scaler.update()
            for k, t in parts.items():
                agg[k] = agg.get(k, 0.0) + float(t.detach())
            agg["n"] = agg.get("n", 0) + 1
        n = agg.pop("n", 1)
        return {k: round(t / n, 4) for k, t in agg.items()}

    for ep in range(1, E + 1):
        stage = "warm" if ep <= cfg["ep_warm"] else "joint"
        _set_trainable(model, stage)
        t0 = time.time()
        tl = one_epoch(opt_warm if stage == "warm" else opt_joint, stage)
        if stage == "joint": sched.step()
        pv = predict(model, va_loader, device)
        auc = auc_safe(pv["y"], pv["prob"])
        rec = dict(epoch=ep, stage=stage, val_auc=round(auc, 4), sec=round(time.time() - t0, 1), **tl)
        hist.append(rec); log(f"[{cfg['name']} s{cfg['seed']}] {rec}")
        if not is_proto and auc > best["auc"]:
            best = dict(auc=auc, state=copy.deepcopy({k: t.cpu() for k, t in model.state_dict().items()}), epoch=ep)

    push_info = None
    if is_proto:
        push_info = push_prototypes(model, tr_eval_loader, device, constrained=cfg["push"] == "constrained")
        pv = predict(model, va_loader, device)
        auc = auc_safe(pv["y"], pv["prob"])
        hist.append(dict(epoch=E, stage="after_push", val_auc=round(auc, 4)))
        log(f"[{cfg['name']} s{cfg['seed']}] after push val_auc={auc:.4f} | protos on artifact cells(>0.5): "
            f"{int((push_info.art_frac > 0.5).sum())}/{len(push_info)} | on lesion cells(>=0.5): "
            f"{int((push_info.les_frac >= 0.5).sum())}/{len(push_info)}")
        best = dict(auc=auc, state=copy.deepcopy({k: t.cpu() for k, t in model.state_dict().items()}), epoch=E)
        _set_trainable(model, "last")
        opt_ll = torch.optim.Adam(model.last.parameters(), lr=cfg["lr_ll"])
        for k in range(1, cfg["ep_ll"] + 1):
            t0 = time.time()
            tl = one_epoch(opt_ll, "last")
            pv = predict(model, va_loader, device)
            auc = auc_safe(pv["y"], pv["prob"])
            rec = dict(epoch=E + k, stage="last", val_auc=round(auc, 4), sec=round(time.time() - t0, 1), **tl)
            hist.append(rec); log(f"[{cfg['name']} s{cfg['seed']}] {rec}")
            if auc > best["auc"]:
                best = dict(auc=auc, state=copy.deepcopy({k2: t.cpu() for k2, t in model.state_dict().items()}), epoch=E + k)
        push_info.to_csv(os.path.join(outdir, "push_info.csv"), index=False)

    if best["state"] is None:   # e.g. undefined val AUC: never save an empty checkpoint
        best = dict(auc=float("nan"), state={k: t.cpu() for k, t in model.state_dict().items()}, epoch=len(hist))
    torch.save(best["state"], os.path.join(outdir, "model.pt"))
    json.dump(dict(cfg=cfg, hist=hist, best_epoch=best["epoch"], best_val_auc=best["auc"],
                   minutes=round((time.time() - t_start) / 60, 1)), open(os.path.join(outdir, "log.json"), "w"), indent=1)
    log(f"[{cfg['name']} s{cfg['seed']}] DONE best_val_auc={best['auc']:.4f} at epoch {best['epoch']} "
        f"({(time.time() - t_start) / 60:.1f} min)")
    return best["auc"]


def find_prep_dirs(files=None):
    """Locate NB00 output (dir with meta.csv + images.npy) and NB00b output (dir with meta.csv + masks.npy +
    dup_candidates.csv), independent of how Kaggle mounts notebook outputs. Prints a diagnosis if missing."""
    files = files if files is not None else walk_files("/kaggle/input") + walk_files("/kaggle/working")
    by_dir = {}
    for f in files:
        by_dir.setdefault(os.path.dirname(f), set()).add(os.path.basename(f))
    p1 = sorted(d for d, fs in by_dir.items() if {"meta.csv", "images.npy"} <= fs)
    p2 = sorted(d for d, fs in by_dir.items() if {"meta.csv", "masks.npy", "dup_candidates.csv"} <= fs)
    if not p1:
        print("!!! NB00 output NOT found under /kaggle/input. What IS mounted (first 3 levels):")
        for d in sorted({"/".join(f.split("/")[:6]) for f in files})[:40]:
            print("   ", d)
        print("   any meta.csv:", [f for f in files if f.endswith("meta.csv")][:10])
    return (p1[0] if p1 else None), (p2[0] if p2 else None)


In [ ]:
%%writefile prepare.py
import matplotlib
matplotlib.use('Agg')
import os, re, glob, json, time, sys, numpy as np, pandas as pd, cv2, torch
from multiprocessing import Pool
import importlib, artproto as ap; importlib.reload(ap)
OUT = '/kaggle/working/prep'; os.makedirs(OUT, exist_ok=True)
T0 = time.time()
print('torch', torch.__version__, '| cuda', torch.cuda.is_available(), '| GPUs', torch.cuda.device_count(), '| CPUs', os.cpu_count())
for p in sorted(glob.glob('/kaggle/input/*')) + sorted(glob.glob('/kaggle/input/*/*'))[:40]:
    print(('LINK -> ' + os.path.realpath(p)) if os.path.islink(p) else ('DIR ' if os.path.isdir(p) else 'FILE'), p)
ALL = ap.walk_files('/kaggle/input')
print(len(ALL), 'files under /kaggle/input')
print('top-level inputs:', sorted({'/'.join(p.split('/')[:5]) for p in ALL})[:30])
assert ALL, "NO FILES under /kaggle/input: add the datasets via Add Input (right panel) and re-run"
print('artifact detector version:', ap.DETECTOR_VERSION)

# ---------------- next cell ----------------
gt_csv = [f for f in ALL if os.path.basename(f) == 'ISIC_2019_Training_GroundTruth.csv']
md_csv = [f for f in ALL if os.path.basename(f) == 'ISIC_2019_Training_Metadata.csv']
assert gt_csv and md_csv, 'ISIC 2019 csv files not found'
gt, mt = pd.read_csv(gt_csv[0]), pd.read_csv(md_csv[0])
df = gt.merge(mt, on='image', how='left')
img_map = {}
for f in ALL:
    b = os.path.basename(f)
    # only take images from the official ISIC-2019 training folder (other inputs may contain copies of HAM images)
    if b.lower().endswith('.jpg') and b.startswith('ISIC_') and 'isic_2019_training_input' in f.lower():
        img_map[os.path.splitext(b)[0]] = f
df['path'] = df.image.map(img_map)
print('rows', len(df), '| with image file', df.path.notna().sum())
df = df[df.path.notna()].reset_index(drop=True)
df['label'] = df.MEL.astype(int)

def source_of(r):
    lid = r['lesion_id']
    if isinstance(lid, str):
        for s in ('HAM', 'BCN', 'MSK'):
            if lid.startswith(s): return s
    n = int(re.search(r'ISIC_(\d{7})', r['image']).group(1))
    if 24306 <= n <= 34320: return 'HAM'
    if n < 24306 or 'downsampled' in r['image']: return 'MSK'
    return 'BCN'
df['source'] = df.apply(source_of, axis=1)
print(pd.crosstab(df.source, df.label, margins=True))
print('missing lesion_id by source:\n', df[df.lesion_id.isna()].source.value_counts())

seg_map = {}
for f in ALL:   # accepts ISIC_xxxxxxx_segmentation.png, or ISIC_xxxxxxx.png inside a *mask*/*seg* folder
    b, low = os.path.basename(f), f.lower()
    m = re.match(r'^(ISIC_\d{7})(_segmentation|_mask|_seg)?\.(png|bmp|jpg)$', b, re.I)
    if not m or 'isic_2019_training_input' in low: continue
    parent = os.path.basename(os.path.dirname(f)).lower()
    if m.group(2) or (b.lower().endswith(('.png', '.bmp')) and any(k in parent for k in ('mask', 'seg', 'groundtruth'))):
        if m.group(1) not in seg_map or m.group(2): seg_map[m.group(1)] = f
print('example mask files:', list(seg_map.values())[:3])
if seg_map:
    from PIL import Image as _I
    _u = np.unique(np.asarray(_I.open(next(iter(seg_map.values()))).convert('L')))
    print('values in one mask file (must be binary-like, e.g. [0 255]):', _u[:10], '... n_unique =', len(_u))
df['gt_mask_path'] = df.image.str.slice(0, 12).map(seg_map)
print('GT lesion masks available for', df.gt_mask_path.notna().sum(), 'ISIC-2019 images (seg files found:', len(seg_map), ')')

# ---------------- next cell ----------------
N = len(df)
IM = np.lib.format.open_memmap(f'{OUT}/images.npy', mode='w+', dtype=np.uint8, shape=(N, 256, 256, 3))
GTM = np.lib.format.open_memmap(f'{OUT}/gt_lesion.npy', mode='w+', dtype=np.uint8, shape=(N, 256, 256))
def cache_one(a):
    i, p, mp = a
    try: img, ok = ap.load_rgb_resized(p), True
    except Exception: img, ok = np.zeros((256, 256, 3), np.uint8), False
    gm = None
    if isinstance(mp, str):
        try: gm = ap.load_mask_resized(mp)
        except Exception: gm = None
    tf, tc = ap.thumbs(img)
    return i, img, gm, ok, ap.phash64(img), tf, tc
HB = np.zeros((N, 64), bool); TF = np.zeros((N, 1024), np.float16); TC = np.zeros((N, 1024), np.float16); okv = np.zeros(N, bool); has_gt = np.zeros(N, bool)
t = time.time()
with Pool(os.cpu_count()) as pool:
    for k, (i, img, gm, ok, hb, tf, tc) in enumerate(pool.imap_unordered(cache_one, zip(range(N), df.path, df.gt_mask_path), chunksize=16)):
        IM[i] = img; okv[i] = ok; HB[i] = hb; TF[i] = tf; TC[i] = tc
        if gm is not None: GTM[i] = gm; has_gt[i] = True
        if k % 5000 == 0: print(k, f'{time.time()-t:.0f}s', flush=True)
IM.flush(); GTM.flush()
df['decode_ok'] = okv; df['has_gt_mask'] = has_gt
print('decode failures:', (~okv).sum(), '| GT masks loaded:', has_gt.sum(), f'| {time.time()-t:.0f}s')

# ---------------- next cell ----------------
ii, jj, hh = ap.hamming_pairs(HB, thr=8)
cf = np.einsum('ij,ij->i', TF[ii].astype(np.float32), TF[jj].astype(np.float32))
cc = np.einsum('ij,ij->i', TC[ii].astype(np.float32), TC[jj].astype(np.float32))
lid = df.lesion_id.fillna('NA_' + df.image).values
same = lid[ii] == lid[jj]
print('pHash candidates (Hamming<=8):', len(ii), '| of which same lesion_id:', same.sum())
for name, msk in (('same lesion_id', same), ('different/missing lesion_id', ~same)):
    if msk.sum():
        print(f'  centre-crop correlation, {name}: quantiles', np.round(np.quantile(cc[msk], [.05, .25, .5, .75, .95]), 3))
THR = 0.90
dup = (cf >= THR) & (cc >= THR)
print(f'VERIFIED duplicates (full & centre correlation >= {THR}): {dup.sum()} pairs | '
      f'spanning different lesion_ids: {(dup & ~same).sum()} | involving images without lesion_id: '
      f'{(dup & (df.lesion_id.isna().values[ii] | df.lesion_id.isna().values[jj])).sum()}')
pd.DataFrame(dict(i=ii, j=jj, ham=hh, corr_full=cf, corr_centre=cc, same_lesion=same, verified=dup)).to_csv(f'{OUT}/dup_candidates.csv', index=False)

# ---------------- next cell ----------------
import matplotlib.pyplot as plt
rs = np.random.RandomState(0)
for title, sel in (('VERIFIED duplicates, different lesion_id', np.where(dup & ~same)[0]),
                   ('REJECTED candidates (pHash<=8 but low correlation)', np.where(~dup & ~same)[0])):
    if len(sel) == 0: continue
    pick = rs.choice(sel, min(6, len(sel)), replace=False)
    fig, ax = plt.subplots(2, len(pick), figsize=(3 * len(pick), 6))
    ax = np.array(ax).reshape(2, -1)
    for c, p in enumerate(pick):
        for r, k in enumerate((ii[p], jj[p])):
            ax[r, c].imshow(IM[k]); ax[r, c].axis('off')
            ax[r, c].set_title(f'{df.image[k]}\n{lid[k][:12]} cc={cc[p]:.2f}', fontsize=7)
    fig.suptitle(title); plt.tight_layout(); plt.savefig(f'{OUT}/qc_dups_{"verified" if "VER" in title else "rejected"}.png', dpi=90); plt.show()

# ---------------- next cell ----------------
from sklearn.model_selection import StratifiedGroupKFold
uf = ap.UnionFind(N)
for _, idx in df.groupby('lesion_id').indices.items():
    for a in idx[1:]: uf.union(idx[0], a)
for a, b in zip(ii[dup], jj[dup]): uf.union(a, b)
df['group'] = [uf.find(k) for k in range(N)]
sz = df.group.value_counts()
print('groups', len(sz), '| largest', sz.iloc[0], '| group-size quantiles', sz.quantile([.5, .9, .99]).to_dict())
sgkf = StratifiedGroupKFold(n_splits=10, shuffle=True, random_state=42)
fold = np.zeros(N, int)
for f, (_, te) in enumerate(sgkf.split(np.zeros(N), df.label, groups=df.group)): fold[te] = f
df['split'] = np.where(fold == 0, 'test', np.where(fold == 1, 'val', 'train'))
for a in ('train', 'val', 'test'):
    for b in ('train', 'val', 'test'):
        if a < b: assert not set(df.group[df.split == a]) & set(df.group[df.split == b])
for a in ('train', 'val', 'test'):     # independent check on lesion_id itself
    for b in ('train', 'val', 'test'):
        if a < b: assert not set(df.lesion_id[(df.split == a) & df.lesion_id.notna()]) & set(df.lesion_id[(df.split == b) & df.lesion_id.notna()])
vd = np.where(dup)[0]
assert all(df.split[ii[p]] == df.split[jj[p]] for p in vd)
print('leakage checks passed: no shared group, no shared lesion_id, every verified duplicate pair in the same split')
print(pd.crosstab(df.split, df.label, margins=True))
print((pd.crosstab(df.source, df.split, normalize='index') * 100).round(1))
print('melanoma rate by split:', df.groupby('split').label.mean().round(4).to_dict())

# ---------------- next cell ----------------
import torch.nn.functional as F
from torchvision.models.segmentation import deeplabv3_mobilenet_v3_large
from torchvision.models import MobileNet_V3_Large_Weights
dev = torch.device('cuda')
torch.manual_seed(0); np.random.seed(0); torch.backends.cudnn.benchmark = False; torch.backends.cudnn.deterministic = True
gt_idx = np.where(df.has_gt_mask.values)[0]
seg_tr = gt_idx[df.split.values[gt_idx] == 'train']; seg_ev = gt_idx[df.split.values[gt_idx] != 'train']
print('seg train', len(seg_tr), '| seg held-out eval', len(seg_ev))
USE_SEG = len(seg_tr) >= 500
MEAN_T = torch.tensor(ap.MEAN, device=dev).view(1,3,1,1); STD_T = torch.tensor(ap.STD, device=dev).view(1,3,1,1)
IMr = np.load(f'{OUT}/images.npy', mmap_mode='r'); GTr = np.load(f'{OUT}/gt_lesion.npy', mmap_mode='r')
class SegDS(torch.utils.data.Dataset):
    def __init__(s, idx): s.idx = idx
    def __len__(s): return len(s.idx)
    def __getitem__(s, k):
        i = s.idx[k]; return torch.from_numpy(np.array(IMr[i])), torch.from_numpy(np.array(GTr[i]))
def norm(x): return ((x.permute(0,3,1,2).float() / 255) - MEAN_T) / STD_T
def dice_np(a, b): return 2 * (a & b).sum() / max(a.sum() + b.sum(), 1)
if USE_SEG:
    seg = deeplabv3_mobilenet_v3_large(weights=None, weights_backbone=MobileNet_V3_Large_Weights.IMAGENET1K_V1,
                                       num_classes=1, aux_loss=False).to(dev)
    dl = torch.utils.data.DataLoader(SegDS(seg_tr), batch_size=32, shuffle=True, num_workers=4, drop_last=True)
    EP = 12
    opt = torch.optim.AdamW(seg.parameters(), lr=5e-4, weight_decay=1e-4)
    sch = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=5e-4, total_steps=EP * len(dl))
    scaler = torch.amp.GradScaler('cuda')
    for ep in range(EP):
        seg.train(); tl = 0; t = time.time()
        for x, y in dl:
            x, y = norm(x.to(dev)), y.to(dev).float()[:, None]
            if np.random.rand() < .5: x, y = x.flip(3), y.flip(3)
            if np.random.rand() < .5: x, y = x.flip(2), y.flip(2)
            k = np.random.randint(4); x, y = torch.rot90(x, k, (2, 3)), torch.rot90(y, k, (2, 3))
            with torch.autocast('cuda', dtype=torch.float16):
                lo = seg(x)['out'].float()
            p = torch.sigmoid(lo)
            loss = F.binary_cross_entropy_with_logits(lo, y) + 1 - (2 * (p * y).sum((1,2,3)) + 1) / (p.sum((1,2,3)) + y.sum((1,2,3)) + 1)
            loss = loss.mean()
            opt.zero_grad(); scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sch.step(); tl += loss.item()
        print(f'seg ep{ep+1} loss {tl/len(dl):.4f} {time.time()-t:.0f}s', flush=True)
    torch.save(seg.state_dict(), f'{OUT}/seg_model.pt')
else:
    print('WARNING: not enough GT masks -> classical Otsu lesion segmentation fallback')

import torch.nn.functional as F
@torch.no_grad()
def seg_predict(arr, idx, bs=64):
    out = np.zeros((len(idx), 256, 256), np.float16)
    if not USE_SEG:
        for k, i in enumerate(idx): out[k] = ap.otsu_lesion(np.array(arr[i]))[0]
        return out
    seg.eval()
    for s in range(0, len(idx), bs):
        x = norm(torch.from_numpy(np.stack([np.array(arr[i]) for i in idx[s:s+bs]])).to(dev))
        with torch.autocast('cuda', dtype=torch.float16):
            p = torch.sigmoid(seg(x)['out'].float()) + torch.sigmoid(seg(x.flip(3))['out'].float()).flip(3)
        out[s:s+len(x)] = (p[:, 0] / 2).cpu().numpy()
    return out
if len(seg_ev):
    pr = seg_predict(IMr, seg_ev)
    d = [dice_np(ap.postprocess_lesion(pr[k].astype(np.float32))[0] > 0, GTr[i] > 0) for k, i in enumerate(seg_ev)]
    j = [x / (2 - x) for x in d]
    print(f'HELD-OUT lesion segmentation: Dice mean {np.mean(d):.4f} median {np.median(d):.4f} | Jaccard mean {np.mean(j):.4f} (n={len(d)})')
    json.dump(dict(dice_mean=float(np.mean(d)), dice_median=float(np.median(d)), jacc_mean=float(np.mean(j)), n=len(d),
                   n_train=int(len(seg_tr))), open(f'{OUT}/seg_eval.json', 'w'))

# ---------------- next cell ----------------
LES = np.lib.format.open_memmap(f'{OUT}/lesion.npy', mode='w+', dtype=np.uint8, shape=(N, 256, 256))
valid = np.zeros(N, bool); src = np.empty(N, object)
t = time.time()
for s in range(0, N, 2048):
    idx = np.arange(s, min(N, s + 2048)); pr = seg_predict(IMr, idx)
    for k, i in enumerate(idx):
        if df.has_gt_mask.values[i]:
            m, v = ap.postprocess_lesion(GTr[i].astype(np.float32)); src[i] = 'gt'
        else:
            m, v = ap.postprocess_lesion(pr[k].astype(np.float32)); src[i] = 'pred'
        LES[i] = m; valid[i] = v
LES.flush()
df['lesion_valid'] = valid; df['lesion_src'] = src
df['lesion_frac'] = [float(LES[i].mean()) for i in range(N)]
print(f'lesion masks done {time.time()-t:.0f}s | valid {valid.mean():.3%} | source', df.lesion_src.value_counts().to_dict())

# ---------------- next cell ----------------
def _init(ip, lp):
    global _I, _L
    _I = np.load(ip, mmap_mode='r'); _L = np.load(lp, mmap_mode='r')
def _art(i):
    return i, ap.detect_artifacts(np.array(_I[i]), np.array(_L[i]))
MS = np.lib.format.open_memmap(f'{OUT}/masks.npy', mode='w+', dtype=np.uint8, shape=(N, 256, 256))
flags = {k: np.zeros(N, bool) for k in ap.ART_BITS}; fr = {k: np.zeros(N) for k in ap.ART_BITS}
t = time.time()
with Pool(os.cpu_count(), initializer=_init, initargs=(f'{OUT}/images.npy', f'{OUT}/lesion.npy')) as pool:
    for i, bits in pool.imap_unordered(_art, range(N), chunksize=32):
        MS[i] = bits | (LES[i] * ap.BIT_L)
        for k, b in ap.ART_BITS.items():
            fr[k][i] = ((bits & b) > 0).mean(); flags[k][i] = fr[k][i] >= ap.ART_MIN_FRAC[k]
MS.flush()
for k in ap.ART_BITS: df[f'art_{k}'] = flags[k]; df[f'artfrac_{k}'] = fr[k]
df['art_any'] = df[[f'art_{k}' for k in ap.ART_BITS]].any(axis=1)
assert ((np.load(f'{OUT}/masks.npy', mmap_mode='r')[:500] & ap.BIT_ART) > 0)[(np.load(f'{OUT}/lesion.npy', mmap_mode='r')[:500] > 0)].sum() == 0
print(f'artifact masks done {time.time()-t:.0f}s (artifact pixels never overlap lesion pixels: verified)')
print(df[[f'art_{k}' for k in ap.ART_BITS] + ['art_any']].mean().round(4))

# ---------------- next cell ----------------
from scipy.stats import fisher_exact
from statsmodels.stats.contingency_tables import StratifiedTable
rows = []; tr = df[df.split == 'train']
for k in list(ap.ART_BITS) + ['any']:
    col = f'art_{k}'
    a = (tr[col] & (tr.label == 1)).sum(); b = (tr[col] & (tr.label == 0)).sum()
    c = (~tr[col] & (tr.label == 1)).sum(); d = (~tr[col] & (tr.label == 0)).sum()
    orr, p = fisher_exact([[a, b], [c, d]])
    tabs = []
    for s_, g_ in tr.groupby('source'):
        t2 = np.array([[(g_[col] & (g_.label == 1)).sum(), (g_[col] & (g_.label == 0)).sum()],
                       [(~g_[col] & (g_.label == 1)).sum(), (~g_[col] & (g_.label == 0)).sum()]], float)
        if (t2.sum(0) > 0).all() and (t2.sum(1) > 0).all(): tabs.append(t2 + 0.5)
    st = StratifiedTable(tabs) if tabs else None
    ci = st.oddsratio_pooled_confint() if st else (np.nan, np.nan)
    rows.append(dict(artifact=k, n_with=int(a + b), prevalence=round(tr[col].mean(), 4),
                     P_mel_art=round(a / max(a + b, 1), 4), P_mel_noart=round(c / max(c + d, 1), 4),
                     OR=round(orr, 3), p_fisher=f'{p:.2e}',
                     OR_MH=round(st.oddsratio_pooled, 3) if st else np.nan,
                     OR_MH_95CI=f'[{ci[0]:.2f}, {ci[1]:.2f}]',
                     p_MH=f'{max(st.test_null_odds().pvalue, 1e-300):.1e}' if st else 'nan',
                     p_BreslowDay_homog=f'{st.test_equal_odds().pvalue:.2e}' if st else 'nan'))
assoc = pd.DataFrame(rows); print(assoc.to_string()); assoc.to_csv(f'{OUT}/artifact_association_train.csv', index=False)
by_src = []
for (s_, k), g_ in [((s_, k), g_) for s_, g_ in tr.groupby('source') for k in ap.ART_BITS]:
    col = f'art_{k}'; a = (g_[col] & (g_.label == 1)).sum(); b = (g_[col] & (g_.label == 0)).sum()
    c = (~g_[col] & (g_.label == 1)).sum(); d = (~g_[col] & (g_.label == 0)).sum()
    by_src.append(dict(source=s_, artifact=k, prevalence=round(g_[col].mean(), 3),
                       OR=round(((a + .5) * (d + .5)) / ((b + .5) * (c + .5)), 2)))
bs = pd.DataFrame(by_src).pivot(index='artifact', columns='source', values=['prevalence', 'OR']); print('\nper-source:\n', bs)
bs.to_csv(f'{OUT}/artifact_by_source_train.csv')

# ---------------- next cell ----------------
COL = {'DC': (0, 255, 255), 'TS': (255, 255, 0), 'SP': (0, 255, 0), 'INK': (255, 0, 255)}
def overlay(img, m, only=None):
    im = np.array(img).copy()
    for k, b in ap.ART_BITS.items():
        if only and k != only: continue
        cs, _ = cv2.findContours(((m & b) > 0).astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
        cv2.drawContours(im, cs, -1, COL[k], 1)
    cs, _ = cv2.findContours(((m & ap.BIT_L) > 0).astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    cv2.drawContours(im, cs, -1, (255, 0, 0), 2)
    return im
def grid(imgs, masks, ids, title, fn, only=None, ncol=8):
    n = len(ids); nr = int(np.ceil(n / ncol))
    if n == 0: print('no images for', title); return
    fig, ax = plt.subplots(nr, ncol, figsize=(2.6 * ncol, 2.75 * nr)); ax = np.array(ax).reshape(nr, ncol)
    for a in ax.flat: a.axis('off')
    for a, (i, t) in zip(ax.flat, ids): a.imshow(overlay(imgs[i], np.array(masks[i]), only)); a.set_title(t, fontsize=7)
    fig.suptitle(title); plt.tight_layout(); plt.savefig(f'{OUT}/{fn}', dpi=80); plt.show()
rs = np.random.RandomState(1)
for k in ap.ART_BITS:
    sel = np.where(df[f'art_{k}'])[0]
    pick = rs.choice(sel, min(24, len(sel)), replace=False)
    grid(IM, MS, [(i, f'#{c} {df.source[i]} y={df.label[i]}') for c, i in enumerate(pick)],
         f'{k} (v2): {ap.ART_LONG[k]} — only {k} contours + lesion (red)', f'qc2_{k}.png', only=k)
sel = np.where(~df.art_any)[0]; pick = rs.choice(sel, 16, replace=False)
grid(IM, MS, [(i, f'#{c} {df.source[i]} y={df.label[i]}') for c, i in enumerate(pick)], 'no artifact flagged (all contours)', 'qc2_none.png')

# ---------------- next cell ----------------
def ds_root(f):
    # dataset root: /kaggle/input/datasets/<owner>/<name>  (new layout)  or  /kaggle/input/<name>  (old layout)
    p = f.split('/'); i = p.index('input')
    return '/'.join(p[:i + 4]) if p[i + 1] in ('datasets', 'notebooks') else '/'.join(p[:i + 2])

def parse_ph2_labels():
    lab = {}
    for f in [f for f in ALL if 'ph2' in os.path.basename(f).lower() and f.lower().endswith('.txt')]:
        hdr = None
        for line in open(f, errors='ignore'):
            parts = [p.strip() for p in line.split('||')]
            if any('Clinical' in p for p in parts): hdr = parts; continue
            if hdr and len(parts) > 3 and re.match(r'IMD\d{3}', parts[1] if len(parts) > 1 else ''):
                ci = [k for k, p in enumerate(hdr) if 'Clinical' in p][0]
                try: lab[parts[1][:6]] = int(parts[ci])
                except Exception: pass
        if lab: return {k: int(v == 2) for k, v in lab.items()}, 'txt'
    for f in [f for f in ALL if 'ph2' in os.path.basename(f).lower() and f.lower().endswith(('.xlsx', '.xls'))]:
        raw = pd.read_excel(f, header=None).astype(str)
        cells = [(r, c) for r in range(len(raw)) for c in range(raw.shape[1])]
        name_c = [c for r, c in cells if raw.iat[r, c].strip() == 'Image Name'][0]
        mel_c = [c for r, c in cells if raw.iat[r, c].strip().lower() == 'melanoma'][0]   # works with 2-row headers
        out = {raw.iat[r, name_c].strip()[:6]: int(raw.iat[r, mel_c].strip().upper() == 'X')
               for r in range(len(raw)) if raw.iat[r, name_c].strip().startswith('IMD')}
        if out: return out, 'xlsx'
    return None, None

EXT = {}
ph2 = {m.group(1).upper(): f for f in ALL for m in [re.match(r'^(IMD\d{3})\.(bmp|png|jpg)$', os.path.basename(f), re.I)] if m}
if ph2:
    ph2m = {m.group(1).upper(): f for f in ALL for m in [re.match(r'^(IMD\d{3})_lesion\.(bmp|png)$', os.path.basename(f), re.I)] if m}
    labs, how = parse_ph2_labels()
    if labs is None:   # class sub-folders (some mirrors): use them only if they exist, never guess silently
        cls = {k: ('melanoma' in f.lower(), 'nev' in f.lower()) for k, f in ph2.items()}
        if any(a for a, _ in cls.values()) and all(a or b for a, b in cls.values()):
            labs = {k: int(a) for k, (a, b) in cls.items()}; how = 'class folders'
    ids = sorted(ph2)
    if labs is None:   # keep PH2 for EXTERNAL SEGMENTATION validation (expert masks), but never invent labels
        print('!!! PH2: no diagnosis labels found -> PH2 used ONLY for segmentation validation (label=-1)')
        labs, how = {}, 'none'
    EXT['PH2'] = pd.DataFrame(dict(image=ids, path=[ph2[k] for k in ids], label=[labs.get(k, -1) for k in ids],
                                   gt_mask_path=[ph2m.get(k) for k in ids]))
    print('PH2 labels parsed from', how)
for f in [f for f in ALL if os.path.basename(f) == 'meta.csv']:
    m = pd.read_csv(f)
    if {'diagnosis', 'derm', 'clinic'} <= set(m.columns):
        root = ds_root(f)
        idx = {}
        for p in ALL:
            if p.startswith(root): idx['/'.join(p.lower().split('/')[-2:])] = p
        for col, name in (('derm', 'Derm7pt-derm'), ('clinic', 'Derm7pt-clinic')):
            if col == 'derm': print('Derm7pt diagnosis counts:', m.diagnosis.value_counts().to_string())
            e = pd.DataFrame(dict(image=m[col], path=m[col].str.lower().map(idx),
                                  label=m.diagnosis.str.contains('melanoma', case=False).astype(int)))
            e['gt_mask_path'] = None; EXT[name] = e[e.path.notna()].reset_index(drop=True)
        break
for f in [f for f in ALL if f.lower().endswith('.csv')]:
    try: m = pd.read_csv(f, nrows=5)
    except Exception: continue
    if {'img_id', 'diagnostic'} <= set(m.columns):
        m = pd.read_csv(f); root = ds_root(f)
        print('PAD-UFES-20 diagnostic counts:', m.diagnostic.value_counts().to_dict())
        idx = {os.path.basename(p): p for p in ALL if p.startswith(root)}
        e = pd.DataFrame(dict(image=m.img_id, path=m.img_id.map(idx), label=(m.diagnostic == 'MEL').astype(int)))
        e['gt_mask_path'] = None; EXT['PAD-UFES-20'] = e[e.path.notna()].reset_index(drop=True); break
EXPECTED = {'PH2': (200, 40), 'Derm7pt-derm': (1011, None), 'Derm7pt-clinic': (1011, None), 'PAD-UFES-20': (2298, 52)}
for k, e in EXT.items():
    n_exp, m_exp = EXPECTED[k]; mel = int((e.label == 1).sum())
    ok = (len(e) == n_exp) and (m_exp is None or mel == m_exp)
    print(f'{k}: n={len(e)} (expected {n_exp}) melanoma={mel} (expected {m_exp}) unlabeled={int((e.label<0).sum())} '
          f'gt_masks={int(e.gt_mask_path.notna().sum())}  ->', 'OK' if ok else '!!! CHECK')
if not EXT: print('WARNING: no external dataset found — add PH2 / derm7pt / PAD-UFES-20 inputs')

# ---------------- next cell ----------------
ext_summary = {}
tr_idx = np.where(df.split.values == 'train')[0]
for name, e in EXT.items():
    e = e.reset_index(drop=True); n = len(e)    # label = -1 -> unlabeled (segmentation validation only)
    tag = name.replace(' ', '_')
    EI = np.lib.format.open_memmap(f'{OUT}/ext_{tag}_images.npy', mode='w+', dtype=np.uint8, shape=(n, 256, 256, 3))
    EM = np.lib.format.open_memmap(f'{OUT}/ext_{tag}_masks.npy', mode='w+', dtype=np.uint8, shape=(n, 256, 256))
    hb = np.zeros((n, 64), bool); tf = np.zeros((n, 1024), np.float16); tc = np.zeros((n, 1024), np.float16); gtl = [None] * n
    for i, (p, mp) in enumerate(zip(e.path, e.gt_mask_path)):
        EI[i] = ap.load_rgb_resized(p); hb[i] = ap.phash64(EI[i]); tf[i], tc[i] = ap.thumbs(np.array(EI[i]))
        if isinstance(mp, str): gtl[i] = ap.load_mask_resized(mp)
    pr = seg_predict(EI, np.arange(n))
    les_valid, les_src, dices = [], [], []
    for i in range(n):
        pm, pv = ap.postprocess_lesion(pr[i].astype(np.float32))
        if gtl[i] is not None:
            dices.append(dice_np(pm > 0, gtl[i] > 0)); m, v = ap.postprocess_lesion(gtl[i].astype(np.float32)); s = 'gt'
        else: m, v, s = pm, pv, 'pred'
        EM[i] = ap.detect_artifacts(np.array(EI[i]), m) | (m * ap.BIT_L); les_valid.append(v); les_src.append(s)
    EI.flush(); EM.flush()
    e['lesion_valid'] = les_valid; e['lesion_src'] = les_src
    for k, b in ap.ART_BITS.items(): e[f'art_{k}'] = [((EM[i] & b) > 0).mean() >= ap.ART_MIN_FRAC[k] for i in range(n)]
    di, dj, _ = ap.hamming_pairs(hb, HB[tr_idx], thr=8)
    if len(di):
        c1 = np.einsum('ij,ij->i', tf[di].astype(np.float32), TF[tr_idx[dj]].astype(np.float32))
        c2 = np.einsum('ij,ij->i', tc[di].astype(np.float32), TC[tr_idx[dj]].astype(np.float32))
        ver = (c1 >= THR) & (c2 >= THR)
    else: ver = np.zeros(0, bool)
    e['dup_with_isic_train'] = np.isin(np.arange(n), di[ver])
    e.drop(columns=['path', 'gt_mask_path']).to_csv(f'{OUT}/ext_{tag}_meta.csv', index=False)
    ext_summary[name] = dict(n=n, labeled=int((e.label >= 0).sum()), melanoma=int((e.label == 1).sum()),
                             pHash_candidates=int(len(np.unique(di))), verified_dups_with_isic_train=int(e.dup_with_isic_train.sum()),
                             seg_dice_vs_gt=float(np.mean(dices)) if dices else None,
                             **{f'prev_{k}': round(float(e[f'art_{k}'].mean()), 3) for k in ap.ART_BITS})
    print(name, ext_summary[name], flush=True)
    pick = np.random.RandomState(2).choice(n, min(16, n), replace=False)
    grid(EI, EM, [(i, f'{name[:12]} #{c} y={e.label[i]}') for c, i in enumerate(pick)], f'{name}: masks (all types)', f'qc2_ext_{tag}.png')
json.dump(ext_summary, open(f'{OUT}/ext_summary.json', 'w'), indent=1)

# ---------------- next cell ----------------
df.drop(columns=['path', 'gt_mask_path']).to_csv(f'{OUT}/meta.csv', index=False)
del GTM, GTr; import gc; gc.collect()
del LES; gc.collect()
for _f in ('gt_lesion.npy', 'lesion.npy'):      # lesion mask is stored as bit 0 of masks.npy
    try: os.remove(f'{OUT}/{_f}')
    except Exception as ex: print('could not delete', _f, ex)
summ = dict(n=int(N), detector_version=ap.DETECTOR_VERSION, dup_threshold=THR, verified_dup_pairs=int(dup.sum()), largest_group=int(df.group.value_counts().iloc[0]), split=df.split.value_counts().to_dict(), mel_by_split=df.groupby('split').label.mean().round(4).to_dict(),
            lesion_valid=float(df.lesion_valid.mean()), artifact_prev=df[[f'art_{k}' for k in ap.ART_BITS] + ['art_any']].mean().round(4).to_dict(),
            minutes=round((time.time() - T0) / 60, 1))
json.dump(summ, open(f'{OUT}/summary.json', 'w'), indent=1); print(json.dumps(summ, indent=1))
os.system(f'du -sh {OUT}; ls -la {OUT}')

In [ ]:
%%writefile train.py
import argparse, os, sys, glob, json, traceback, pandas as pd, torch
import artproto as ap
p = argparse.ArgumentParser()
p.add_argument('--jobs', required=True)             # ordered by priority: "Ours:0,PPN:0,..." (shared by all GPUs)
p.add_argument('--deadline', type=float, default=0)  # unix time; a job is not started if it cannot finish before it
p.add_argument('--out', default='/kaggle/working/runs')
p.add_argument('--smoke', type=int, default=0)
p.add_argument('--workers', type=int, default=3)
a = p.parse_args()
FILES = ap.walk_files('/kaggle/input') + ap.walk_files('/kaggle/working')
p1, p2 = ap.find_prep_dirs(FILES)                      # NB00: images | NB00b: masks v2 + final split
if not p1: sys.exit('ABORT: NB00 output not found (see diagnosis above)')
p2 = [p2] if p2 else []
if not p2 and not a.smoke:
    sys.exit('ABORT: NB00b output (prep2) not found -- add it as input. Real training must use the final split/masks.')
mdir = p2[0] if p2 else p1
meta = pd.read_csv(f'{mdir}/meta.csv')
prep = dict(img_path=f'{p1}/images.npy', msk_path=f'{mdir}/masks.npy', meta=meta)
dev = torch.device('cuda')
print('images:', p1, '| masks+split:', mdir, '| split sizes:', meta.split.value_counts().to_dict(), '| GPU:', torch.cuda.get_device_name(0), flush=True)
import time
claim_dir = f'{a.out}/claims' + ('_smoke' if a.smoke else ''); os.makedirs(claim_dir, exist_ok=True)
durations = {'cnn': [], 'proto': []}
for job in a.jobs.split(','):
    name, seed = job.split(':')
    out = f'{a.out}/{name}_s{seed}' + ('_smoke' if a.smoke else '')
    if os.path.exists(f'{out}/model.pt'):
        continue
    kind = ap.full_cfg(name, int(seed))['model']
    est = 1.15 * max(durations[kind]) if durations[kind] else (45 if kind == 'cnn' else 75) * 60
    if a.deadline and time.time() + est > a.deadline:
        print(f'DEADLINE: not starting {job} (estimated {est/60:.0f} min, {(a.deadline-time.time())/60:.0f} min left)', flush=True); continue
    try:                                    # atomic claim: exactly one GPU process runs each job
        os.close(os.open(f'{claim_dir}/{name}_s{seed}', os.O_CREAT | os.O_EXCL | os.O_WRONLY))
    except FileExistsError:
        continue
    t0 = time.time()
    print(f'=== START {job} on {torch.cuda.get_device_name(0)} ===', flush=True)
    try:
        ap.train_run(ap.full_cfg(name, int(seed)), prep, out, dev, smoke=bool(a.smoke), workers=a.workers,
                     log=lambda s: print(s, flush=True))
        durations[kind].append(time.time() - t0)
    except Exception:
        traceback.print_exc(); sys.stdout.flush()
print('QUEUE FINISHED for this GPU', flush=True)

In [ ]:
import os, subprocess, time, artproto as ap
T_NOTEBOOK = globals().get('T_NOTEBOOK', time.time())
p1, p2 = ap.find_prep_dirs(ap.walk_files('/kaggle/input') + ap.walk_files('/kaggle/working'))
if p1 and p2:
    print('prepared data found:', p1)
else:
    t0 = time.time(); log = open('/kaggle/working/prepare.log', 'w')
    pr = subprocess.Popen(['python', 'prepare.py'], stdout=log, stderr=subprocess.STDOUT)
    while pr.poll() is None:
        time.sleep(60)
        tail = open('/kaggle/working/prepare.log').read().strip().splitlines()[-2:]
        print(f'[prepare {(time.time()-t0)/60:.0f} min]', *tail, sep='\n   ', flush=True)
    print(open('/kaggle/working/prepare.log').read())
    assert pr.returncode == 0, 'prepare.py failed (see log above)'
    print('PREPARATION DONE in', round((time.time()-t0)/60, 1), 'min')

In [ ]:
import os, subprocess, time, json, glob, torch, pandas as pd
SMOKE = 1                                   # 1 = quick functional test; 0 = real training (Save & Run All)
MAIN = ['Ours', 'PPN', 'PPN-FA', 'CNN', 'PPN-std']          # main comparison (3 seeds)
ABL = ['Ours-noLes', 'Ours-noArt', 'Ours-stdPush']          # ablations
jobs = [f'{n}:{s}' for s in (0, 1, 2) for n in MAIN] + [f'{n}:{s}' for s in (0, 1, 2) for n in ABL]   # priority order
if SMOKE: jobs = ['Ours:0', 'PPN-FA:0', 'CNN:0']
T_START = globals().get('T_NOTEBOOK', time.time())
DEADLINE = 0 if SMOKE else T_START + 11.2 * 3600            # Kaggle commit limit is 12 h; keep margin for saving outputs
ngpu = max(1, torch.cuda.device_count())
os.makedirs('/kaggle/working/runs', exist_ok=True)
procs = []
for g in range(ngpu):
    log = open(f'/kaggle/working/runs/gpu{g}.log', 'a')
    env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(g))
    procs.append((subprocess.Popen(['python', 'train.py', '--jobs', ','.join(jobs), '--smoke', str(SMOKE),
                                    '--deadline', str(DEADLINE)], stdout=log, stderr=subprocess.STDOUT, env=env), g, jobs))
print('shared priority queue:', jobs)
print('deadline:', time.ctime(DEADLINE) if DEADLINE else 'none')
t0 = time.time()
while any(p.poll() is None for p, _, _ in procs):
    time.sleep(30 if SMOKE else 600)
    print(f'--- {(time.time()-t0)/60:.0f} min ---', flush=True)
    for _, g, _ in procs:
        lines = open(f'/kaggle/working/runs/gpu{g}.log').read().strip().splitlines()
        print(f'[GPU{g}]', *lines[-3:], sep='\n  ', flush=True)
print('ALL DONE in', round((time.time()-t0)/60, 1), 'min')

In [ ]:
for g in range(ngpu):
    f = f'/kaggle/working/runs/gpu{g}.log'
    if os.path.exists(f): print(f'===== GPU{g} full log =====\n' + open(f).read())

In [ ]:
rows = []
for f in sorted(glob.glob('/kaggle/working/runs/*/log.json')):
    L = json.load(open(f)); h = pd.DataFrame(L['hist'])
    r = dict(run=os.path.basename(os.path.dirname(f)), best_val_auc=round(L['best_val_auc'], 4), best_epoch=L['best_epoch'],
             minutes=L['minutes'], sec_per_epoch=round(h.sec.dropna().mean(), 1))
    pf = os.path.join(os.path.dirname(f), 'push_info.csv')
    if os.path.exists(pf):
        pi = pd.read_csv(pf); r.update(protos_on_artifact=int((pi.art_frac > 0.5).sum()), protos_on_lesion=int((pi.les_frac >= 0.5).sum()), P=len(pi))
    rows.append(r)
print(pd.DataFrame(rows).to_string())